<a href="https://colab.research.google.com/github/SharareTaheriMoghadam/clinical-readmission-prediction/blob/main/notebook/EDA_Modeling_and_Explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# Clinical Readmission Prediction Project
# Exploratory Data Analysis and Machine Learning Modeling
#
# Purpose:
# - Explore clinical predictors associated with readmission
# - Develop predictive models
# - Compare interpretable and ensemble approaches
# - Generate explainability outputs
#
# Models:
# 1. Logistic Regression
# 2. Random Forest
#
# Explainability:
# SHAP-based interpretation
#
# ==========================================================

print("""
Clinical Readmission Prediction

This notebook documents:
- Data preprocessing
- Exploratory analysis
- Model development
- Performance evaluation
- Explainability analysis

The workflow supports manuscript reproducibility.
""")

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

import shap

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
# Example:
# df = pd.read_csv("../data/readmission_dataset.csv")


# Demonstration structure
df = pd.DataFrame({

    "age":[78,65,72,81,55],
    "previous_admissions":[4,1,3,5,0],
    "polypharmacy":[9,3,7,12,2],
    "renal_impairment":[1,0,1,1,0],
    "length_of_stay":[12,4,8,15,3],
    "comorbidity_score":[6,2,5,8,1],
    "readmission":[1,0,1,1,0]

})


df.head()

In [ ]:
print("Dataset shape:")
print(df.shape)


print("\nMissing values:")
display(
    df.isnull()
    .sum()
    .to_frame("Missing")
)


print("\nData types:")
display(
    df.dtypes
    .to_frame("Type")
)

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x="readmission",
    data=df
)

plt.title(
    "Distribution of Readmission Outcome"
)

plt.xlabel(
    "Readmission"
)

plt.ylabel(
    "Number of Patients"
)

plt.show()

In [ ]:
clinical_features = [
    "age",
    "previous_admissions",
    "polypharmacy",
    "renal_impairment",
    "length_of_stay",
    "comorbidity_score"
]


df[clinical_features].hist(
    figsize=(12,8)
)

plt.suptitle(
    "Distribution of Clinical Predictors"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,6))


sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm"
)


plt.title(
    "Correlation Between Clinical Features"
)


plt.show()

In [ ]:
X = df.drop(
    "readmission",
    axis=1
)

y = df["readmission"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)


print(
    X_train.shape,
    X_test.shape
)

In [ ]:
numeric_features = X.columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [ ]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                random_state=RANDOM_STATE
            )
        )
    ]
)


logistic_model.fit(
    X_train,
    y_train
)


lr_prediction = logistic_model.predict(
    X_test
)


lr_probability = logistic_model.predict_proba(
    X_test
)[:,1]

In [ ]:
rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE
            )
        )
    ]
)


rf_model.fit(
    X_train,
    y_train
)


rf_prediction = rf_model.predict(
    X_test
)


rf_probability = rf_model.predict_proba(
    X_test
)[:,1]

In [ ]:
results = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy":[
        accuracy_score(y_test, lr_prediction),
        accuracy_score(y_test, rf_prediction)
    ],

    "Precision":[
        precision_score(y_test, lr_prediction),
        precision_score(y_test, rf_prediction)
    ],

    "Recall":[
        recall_score(y_test, lr_prediction),
        recall_score(y_test, rf_prediction)
    ],

    "F1":[
        f1_score(y_test, lr_prediction),
        f1_score(y_test, rf_prediction)
    ],

    "AUROC":[
        roc_auc_score(y_test, lr_probability),
        roc_auc_score(y_test, rf_probability)
    ]

})


results

In [ ]:
plt.figure(figsize=(7,5))


for name,prob in [

    ("Logistic Regression",
     lr_probability),

    ("Random Forest",
     rf_probability)

]:

    fpr,tpr,_ = roc_curve(
        y_test,
        prob
    )

    auc = roc_auc_score(
        y_test,
        prob
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} AUC={auc:.3f}"
    )


plt.plot(
    [0,1],
    [0,1],
    "--"
)


plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve Comparison"
)


plt.legend()

plt.show()

In [ ]:
coefficients = pd.DataFrame({

    "Feature":
        X.columns,

    "Coefficient":
        logistic_model
        .named_steps["classifier"]
        .coef_[0]

})


coefficients = (
    coefficients
    .sort_values(
        "Coefficient"
    )
)


plt.figure(figsize=(8,5))


plt.barh(
    coefficients.Feature,
    coefficients.Coefficient
)


plt.title(
    "Logistic Regression Feature Contributions"
)


plt.xlabel(
    "Coefficient"
)


plt.show()

In [ ]:
importance = pd.DataFrame({

    "Feature":
        X.columns,

    "Importance":
        rf_model
        .named_steps["classifier"]
        .feature_importances_

})


importance = importance.sort_values(
    "Importance",
    ascending=False
)


plt.figure(figsize=(8,5))


plt.barh(
    importance.Feature,
    importance.Importance
)


plt.gca().invert_yaxis()


plt.title(
    "Random Forest Feature Importance"
)


plt.xlabel(
    "Importance"
)


plt.show()

In [ ]:
# Transform test data

X_test_processed = (
    rf_model
    .named_steps["preprocessor"]
    .transform(X_test)
)


explainer = shap.TreeExplainer(
    rf_model
    .named_steps["classifier"]
)


shap_values = explainer(
    X_test_processed
)


shap.summary_plot(
    shap_values[:,:,1],
    X_test,
    feature_names=X.columns
)

In [ ]:
import os

os.makedirs(
    "../figures",
    exist_ok=True
)


plt.savefig(
    "../figures/model_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)